# MPMAvatar — SDS Physics Training
Run cells top to bottom. Each section is self-contained.

## Cell 1 — Check GPU

In [ ]:
import subprocess, os, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

cc = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{cc[0]}.{cc[1]}"
os.environ['FORCE_CUDA'] = '1'
print(f"CUDA arch: sm_{cc[0]}{cc[1]}")


## Cell 2 — Clone Repo

In [ ]:
import os

REPO_DIR = "/teamspace/studios/this_studio/mpm_avatar_vds"
GITHUB_USER = "poudelsaroj"   # change if needed

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/mpm_avatar_vds.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git stash && git pull
print("Repo ready:", REPO_DIR)


## Cell 3 — Install Python Dependencies
Run once per session. Uses numpy 2.x throughout (Wan works fine despite the version warning).

In [ ]:
# Core deps — keep numpy 2.x, do NOT downgrade (Wan works with 2.x at runtime)
!pip install -q \
    "numpy>=2.0" \
    scipy Pillow tqdm pyyaml wandb einops plyfile trimesh \
    smplx roma jaxtyping mediapy imageio safetensors accelerate \
    "warp-lang>=1.0.0" \
    transformers sentencepiece huggingface_hub \
    human-body-prior \
    librosa opencv-python-headless peft easydict ftfy \
    diffusers decord tokenizers imageio-ffmpeg

import numpy as np, sys
print(f"numpy {np.__version__} / Python {sys.version.split()[0]}")
print("Done. Run next cell to install pytorch3d.")


## Cell 4 — Install pytorch3d (~5 min, builds from source)

In [ ]:
import os, torch

# Detect GPU arch — only compile for the current card (halves build time)
cc = torch.cuda.get_device_capability()
arch = f"{cc[0]}.{cc[1]}"
os.environ["TORCH_CUDA_ARCH_LIST"] = arch
os.environ["FORCE_CUDA"] = "1"
os.environ["MAX_JOBS"]   = "8"   # parallel C++ jobs — ~10 min on L4, ~3 min on H100
print(f"Building pytorch3d for sm_{cc[0]}{cc[1]} with MAX_JOBS=8 ...")

!pip install -q fvcore iopath
!pip install -q "git+https://github.com/facebookresearch/pytorch3d.git@stable" --no-build-isolation

import pytorch3d
print(f"✓ pytorch3d {pytorch3d.__version__}")

## Cell 5 — Build CUDA Extensions (diff_gauss, simple_knn)

In [ ]:
import os, subprocess, sys

REPO_DIR = "/teamspace/studios/this_studio/mpm_avatar_vds"
EXT_DIR  = "/teamspace/studios/this_studio/ext"
os.makedirs(EXT_DIR, exist_ok=True)

def build(name, repo_url, extra_patch=None):
    dst = f"{EXT_DIR}/{name}"
    if not os.path.exists(dst):
        subprocess.run(["git","clone","--recurse-submodules", repo_url, dst], check=True)
    if extra_patch:
        extra_patch(dst)
    subprocess.run([sys.executable,"-m","pip","install","-e",dst,"--no-build-isolation","-q"], check=True)
    print(f"{name} done")

def patch_diff_gauss(path):
    # Add missing #include <cstdint>
    f = f"{path}/cuda_rasterizer/auxiliary.h"
    src = open(f).read()
    if "#include <cstdint>" not in src:
        open(f,"w").write("#include <cstdint>\n" + src)

build("diff-gaussian-rasterization",
      "https://github.com/slothfulxtx/diff-gaussian-rasterization.git",
      patch_diff_gauss)
build("simple-knn",
      "https://gitlab.inria.fr/bkerbl/simple-knn.git")

# Verify
subprocess.run([sys.executable,"-c","import diff_gauss; print('✓ diff_gauss OK')"], check=True)
subprocess.run([sys.executable,"-c","import simple_knn; print('✓ simple_knn OK')"], check=True)


## Cell 6 — Verify diffusers WanPipeline (Wan 5B)

In [ ]:
# Upgrade diffusers to latest (needs WanPipeline support added in ~0.32+)
!pip install -q -U "git+https://github.com/huggingface/diffusers"

# Verify WanPipeline is importable
try:
    from diffusers import WanPipeline, AutoencoderKLWan
    import diffusers
    print(f"\u2713 diffusers {diffusers.__version__} — WanPipeline OK")
except ImportError as e:
    print(f"\u2717 WanPipeline not found: {e}")
    print("  Try: pip install git+https://github.com/huggingface/diffusers")


## Cell 7 — Download Wan2.2-TI2V-5B-Diffusers (~22 GB)
Run once, skips if already downloaded.  5B model fits on a single A100/H100 GPU.

In [ ]:
import os
from huggingface_hub import snapshot_download, login

HF_TOKEN     = os.environ.get("HF_TOKEN", "")   # set in Lightning Secrets or paste here
WAN_CKPT_DIR = "/teamspace/studios/this_studio/wan_5b_model"
os.makedirs(WAN_CKPT_DIR, exist_ok=True)

# Check if already downloaded (diffusers layout has model_index.json)
if not os.path.exists(f"{WAN_CKPT_DIR}/model_index.json"):
    print("Downloading Wan2.2-TI2V-5B-Diffusers (~22 GB) ...")
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
    snapshot_download(
        repo_id="Wan-AI/Wan2.2-TI2V-5B-Diffusers",
        local_dir=WAN_CKPT_DIR,
        ignore_patterns=["*.md", "*.txt"],
        token=HF_TOKEN or None,
    )
    print("Download complete.")
else:
    print("Already downloaded.")

print("\nContents:")
for name in sorted(os.listdir(WAN_CKPT_DIR)):
    print(f"  {name}")


## Cell 8 — Verify Wan 5B Loads
Quick smoke test: instantiate the pipeline and check all components are present.

In [ ]:
import os, torch
from diffusers import WanPipeline, AutoencoderKLWan

WAN_CKPT_DIR = "/teamspace/studios/this_studio/wan_5b_model"

if not os.path.exists(f"{WAN_CKPT_DIR}/model_index.json"):
    print("\u2717 Model not found — run Cell 7 first.")
else:
    print("Checking model components (not loading weights yet) ...")
    required = ["transformer", "vae", "text_encoder", "tokenizer", "scheduler", "model_index.json"]
    all_ok = True
    for name in required:
        path = f"{WAN_CKPT_DIR}/{name}"
        ok = os.path.exists(path)
        if not ok:
            all_ok = False
        print(f"  {'\u2713' if ok else '\u2717'} {name}")

    if all_ok:
        print("\n\u2713 All components present — ready for training.")
    else:
        print("\n\u2717 Missing components — re-run Cell 7.")


## Cell 9 — Download & Place Dataset Files

Download the full dataset TAR from Google Drive, then run this cell to place everything.

```bash
# In Lightning terminal:
pip install gdown
gdown "1d1UXaqhiqOUx1NqggKreGrClldb9dxeR" -O /teamspace/studios/this_studio/mpmavatar_data.tar.gz
tar -xzf /teamspace/studios/this_studio/mpmavatar_data.tar.gz -C /teamspace/studios/this_studio/
```

The TAR extracts to `/teamspace/studios/this_studio/data/` which has the full dataset structure.
Run this cell to copy model files to `pretrained_models/`.


In [ ]:
import os, shutil, glob

STUDIO      = "/teamspace/studios/this_studio"
PRETRAINED  = f"{STUDIO}/pretrained_models"
REPO_DIR    = f"{STUDIO}/mpm_avatar_vds"
DATA_DIR    = f"{STUDIO}/data"              # extracted TAR lands here

# Destinations
PLY_DST      = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
AOMAP_DST    = f"{TRACKING_DST}/aomap"

for d in [PLY_DST, AOMAP_DST]:
    os.makedirs(d, exist_ok=True)

def cp(src, dst, label):
    if not src or not os.path.exists(src):
        print(f"  ✗ NOT FOUND  {label}")
        return False
    if os.path.exists(dst):
        print(f"  · exists     {label}")
        return True
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  ✓ copied     {label}")
    return True

# ── Gaussian checkpoint files (from processed data ZIP / mpm_zip_contents) ───
ZIP_BASE = f"{STUDIO}/mpm_zip_contents/MPMAvatar_processed_data"
cp(f"{ZIP_BASE}/model/a1_s1/point_cloud/timestep_030000/point_cloud.ply",  f"{PLY_DST}/point_cloud.ply",  "point_cloud.ply")
cp(f"{ZIP_BASE}/model/a1_s1/point_cloud/timestep_030000/verts_offset.npy", f"{PLY_DST}/verts_offset.npy", "verts_offset.npy")
cp(f"{ZIP_BASE}/model/a1_s1/point_cloud/timestep_030000/cams.npz",         f"{PLY_DST}/cams.npz",         "cams.npz")
cp(f"{ZIP_BASE}/model/a1_s1/point_cloud/timestep_030000/shadow_net.pt",    f"{PLY_DST}/shadow_net.pt",    "shadow_net.pt")

# ── Tracking params + AO maps (from processed data ZIP) ───────────────────────
params_src = f"{ZIP_BASE}/output/tracking/a1_s1_460_200"
if os.path.isdir(params_src):
    n = 0
    for f in glob.glob(f"{params_src}/params_*.npz"):
        dst = f"{TRACKING_DST}/{os.path.basename(f)}"
        if not os.path.exists(dst): shutil.copy2(f, dst); n += 1
    print(f"  ✓ params_*.npz: {n} new ({len(glob.glob(TRACKING_DST+'/params_*.npz'))} total)")
    ao_src = f"{params_src}/aomap"
    if os.path.isdir(ao_src):
        n = 0
        for f in glob.glob(f"{ao_src}/*.png"):
            dst = f"{AOMAP_DST}/{os.path.basename(f)}"
            if not os.path.exists(dst): shutil.copy2(f, dst); n += 1
        print(f"  ✓ aomap/*.png: {n} new ({len(glob.glob(AOMAP_DST+'/*.png'))} total)")
    else:
        print("  ✗ aomap/ not found in ZIP")
else:
    print(f"  ✗ Processed data ZIP not extracted — run mpm_zip_contents extraction first")

# ── Summary check ─────────────────────────────────────────────────────────────
print()
required = {
    "point_cloud.ply":         f"{PLY_DST}/point_cloud.ply",
    "verts_offset.npy":        f"{PLY_DST}/verts_offset.npy",
    "cams.npz":                f"{PLY_DST}/cams.npz",
    "shadow_net.pt":           f"{PLY_DST}/shadow_net.pt",
    "params_460.npz":          f"{TRACKING_DST}/params_460.npz",
    "aomap/mesh_cloth_460.png":f"{AOMAP_DST}/mesh_cloth_460.png",
    "split_idx.npz":           f"{DATA_DIR}/a1_s1/split_idx.npz",
    "cam_info.json":           f"{DATA_DIR}/a1_s1/cam_info.json",
    "optimized_weights.npy":   f"{DATA_DIR}/a1_s1/optimized_weights.npy",
    "SMPLX_NEUTRAL.npz":       f"{DATA_DIR}/body_models/smplx/SMPLX_NEUTRAL.npz",
    "TR00_E096.pt":            f"{DATA_DIR}/body_models/TR00_E096.pt",
    "a1s1_uv.obj":             f"{REPO_DIR}/data/a1_s1/a1s1_uv.obj",
}
all_ok = True
for name, path in required.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {'✓' if ok else '✗'} {name}")
print()
print("✓ All files present!" if all_ok else "✗ Some missing — check above")


## Cell 10 — Generate Missing Data Files
Generates `cam_info.json`, `a1s1_uv.obj`, `smplx_fitted/` and `split_idx.npz` if missing.

In [ ]:
import os, subprocess, sys

REPO_DIR     = "/teamspace/studios/this_studio/mpm_avatar_vds"
PRETRAINED   = "/teamspace/studios/this_studio/pretrained_models"
DATA_DIR     = "/teamspace/studios/this_studio/data"
TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
A1S1_DST     = f"{TRACKING_DST}/a1_s1"

os.makedirs(A1S1_DST, exist_ok=True)
os.chdir(REPO_DIR)

def run(script, *args):
    r = subprocess.run([sys.executable, script, *args], cwd=REPO_DIR)
    return r.returncode == 0

# cam_info.json
cam_info = f"{DATA_DIR}/a1_s1/cam_info.json"
if not os.path.exists(cam_info):
    print("Generating cam_info.json ...")
    run("gen_cam_info.py",
        "--params-dir", f"{TRACKING_DST}",
        "--out", cam_info)

# a1s1_uv.obj
uv_obj = f"{REPO_DIR}/data/a1_s1/a1s1_uv.obj"
if not os.path.exists(uv_obj):
    print("Generating a1s1_uv.obj ...")
    os.makedirs(f"{REPO_DIR}/data/a1_s1", exist_ok=True)
    run("gen_uv_obj.py",
        "--params-dir", f"{TRACKING_DST}",
        "--out", uv_obj)

# split_idx.npz
split_idx = f"{DATA_DIR}/a1_s1/split_idx.npz"
if not os.path.exists(split_idx):
    print("Generating split_idx.npz (synthetic — treats whole mesh as cloth) ...")
    run("gen_split_idx.py")

# smplx_fitted
smplx_dir = f"{DATA_DIR}/a1_s1/smplx_fitted"
if not os.path.exists(smplx_dir) or len(os.listdir(smplx_dir)) == 0:
    print("Generating smplx_fitted ...")
    run("gen_smplx_params.py",
        "--params-dir", f"{TRACKING_DST}",
        "--out-dir", smplx_dir)
    run("gen_smplx_fitted.py",
        "--params-dir", f"{TRACKING_DST}",
        "--out-dir", smplx_dir)

# Verify
for label, path in [
    ("cam_info.json",    cam_info),
    ("a1s1_uv.obj",      uv_obj),
    ("split_idx.npz",    split_idx),
    ("smplx_fitted/",    smplx_dir),
]:
    ok = os.path.exists(path)
    print(f"  {'✓' if ok else '✗'} {label}")


## Cell 11 — Config
Set all paths and training parameters here.

In [ ]:
import os

STUDIO     = "/teamspace/studios/this_studio"
REPO_DIR   = f"{STUDIO}/mpm_avatar_vds"
PRETRAINED = f"{STUDIO}/pretrained_models"

# ── Paths ───────────────────────────────────────────────────────────────────────────────
TRAINED_MODEL_PATH = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
MODEL_PATH         = f"{PRETRAINED}/model/a1_s1"
DATASET_DIR        = f"{STUDIO}/data"
SPLIT_IDX_PATH     = f"{STUDIO}/data/a1_s1/split_idx.npz"

OUTPUT_DIR   = f"{STUDIO}/output"
# Wan2.2-TI2V-5B-Diffusers local snapshot (downloaded in Cell 7)
WAN_CKPT_DIR = f"{STUDIO}/wan_5b_model"
SDS_CFG      = f"{REPO_DIR}/bridge_sds/configs/sds_test.yaml"

# ── Dataset ───────────────────────────────────────────────────────────────────────────
ACTOR             = 1
SEQUENCE          = 1
TRAIN_FRAME_START = 460
TRAIN_FRAME_NUM   = 16  # ≥16 ensures ≥4 VAE latent frames (Wan temporal ×4 compression)
VERTS_START_IDX   = 460

# ── Training ───────────────────────────────────────────────────────────────────────────
ITERATIONS    = 100
WANDB_PROJECT = "MPMAvatar_SDS"
USE_WANDB     = False
WANDB_API_KEY = ""

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(REPO_DIR)

if WANDB_API_KEY:
    import wandb; wandb.login(key=WANDB_API_KEY); print("W&B logged in")

print(f"WAN_CKPT_DIR : {WAN_CKPT_DIR}")
print("Config ready.")


## Cell 12 — Verify All Files Before Training

In [ ]:
import os

checks = {
    "train_sds_physics.py":    f"{REPO_DIR}/train_sds_physics.py",
    "bridge_sds/wan22":        f"{REPO_DIR}/bridge_sds/wan22_i2v_guidance.py",
    "SDS config":              SDS_CFG,
    "point_cloud.ply":         f"{MODEL_PATH}/point_cloud/timestep_030000/point_cloud.ply",
    "verts_offset.npy":        f"{MODEL_PATH}/point_cloud/timestep_030000/verts_offset.npy",
    "cams.npz":                f"{MODEL_PATH}/point_cloud/timestep_030000/cams.npz",
    "shadow_net.pt":           f"{MODEL_PATH}/point_cloud/timestep_030000/shadow_net.pt",
    "params_460.npz":          f"{TRAINED_MODEL_PATH}/params_460.npz",
    "aomap/mesh_cloth_460.png":f"{TRAINED_MODEL_PATH}/aomap/mesh_cloth_460.png",
    "split_idx.npz":           SPLIT_IDX_PATH,
    "cam_info.json":           f"{DATASET_DIR}/a1_s1/cam_info.json",
    "optimized_weights.npy":   f"{DATASET_DIR}/a1_s1/optimized_weights.npy",
    "SMPLX_NEUTRAL.npz":       f"{DATASET_DIR}/body_models/smplx/SMPLX_NEUTRAL.npz",
    "TR00_E096.pt":            f"{DATASET_DIR}/body_models/TR00_E096.pt",
    # Wan 5B diffusers layout
    "Wan model_index.json":    f"{WAN_CKPT_DIR}/model_index.json",
    "Wan transformer/":        f"{WAN_CKPT_DIR}/transformer",
    "Wan vae/":                f"{WAN_CKPT_DIR}/vae",
    "Wan text_encoder/":       f"{WAN_CKPT_DIR}/text_encoder",
    "Wan tokenizer/":          f"{WAN_CKPT_DIR}/tokenizer",
}
all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {'\u2713' if ok else '\u2717'} {name}")
print()
print("\u2713 All checks passed \u2014 ready to train!" if all_ok else "\u2717 Fix missing files before running Cell 13")


## Cell 13 — Run SDS Physics Training

In [ ]:
import os, sys

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
# No wan_repo_root needed — using diffusers WanPipeline directly

wandb_flag     = "--use_wandb" if USE_WANDB else ""
split_idx_flag = f"--split_idx_path {SPLIT_IDX_PATH}" if SPLIT_IDX_PATH and os.path.exists(SPLIT_IDX_PATH) else ""

cmd = (
    f"python train_sds_physics.py "
    f"--trained_model_path {TRAINED_MODEL_PATH} "
    f"--model_path         {MODEL_PATH} "
    f"--dataset_dir        {DATASET_DIR} "
    f"{split_idx_flag} "
    f"--output_dir         {OUTPUT_DIR} "
    f"--actor              {ACTOR} "
    f"--sequence           {SEQUENCE} "
    f"--train_frame_start_num {TRAIN_FRAME_START} {TRAIN_FRAME_NUM} "
    f"--verts_start_idx    {VERTS_START_IDX} "
    f"--wan_ckpt_dir       {WAN_CKPT_DIR} "
    f"--sds_cfg            {SDS_CFG} "
    f"--iterations         {ITERATIONS} "
    f"--save_name          sds_test "
    f"{wandb_flag} "
    f"--wandb_project      {WANDB_PROJECT}"
)

print("Running:\n" + cmd.replace(" --", " \\\n    --"))
print("\n" + "="*60 + "\n")
os.system(cmd)


## Cell 14 — Monitor Training Output

In [ ]:
import os, glob

output_dir = f"{OUTPUT_DIR}/sds_test"
print(f"Output dir: {output_dir}")

csv_files = glob.glob(f"{output_dir}/**/*.csv", recursive=True)
npz_files = glob.glob(f"{output_dir}/**/*.npz", recursive=True)
gif_files = glob.glob(f"{output_dir}/**/*.gif", recursive=True)

print(f"CSV files  : {len(csv_files)}")
print(f"NPZ files  : {len(npz_files)}")
print(f"GIF files  : {len(gif_files)}")

if csv_files:
    import pandas as pd
    df = pd.read_csv(csv_files[-1])
    print(f"\nLatest CSV ({os.path.basename(csv_files[-1])}):")
    print(df.tail(10).to_string())
